# Tarefa 2 — Baseline do Fluxo e Rotulagem (OK, RISCO, FALHA)

**Projeto:** Preditor de degradação de rede com RTT normalizado (independente da rota)  
**Disciplina:** Projeto Integrador / Redes de Computadores  
**Objetivo desta etapa:**
1. Ler os dados brutos consolidados da camada `raw` (pasta `medicoes/`), mantendo o bruto intocado.
2. Separar estritamente o **Período A** (primeiros 7 dias) e o **Período B** (segundos 7 dias), sem compartilhamento de timestamps entre os blocos.
3. Construir a **Ficha de Baseline congelada** (`baseline_por_fluxo.csv`) exclusivamente a partir do Período A, validando o piso de **1.500 amostras válidas** por fluxo.
4. Calcular as métricas relativas normalizadas no Período B (`z_robusto`, `aumento_pct`, `jitter_relativo`, `perda_pct`, `timeout_atual`) e as janelas móveis de persistência (`n5_timeout`, `n5_aumento80`, `n5_risco`).
5. Aplicar a **tabela hierárquica de rotulagem** no Período B (parando estritamente na primeira regra verdadeira): classes **OK**, **RISCO** e **FALHA**.
6. Realizar o **recorte temporal** dentro do Período B (50% Treino, 20% Validação, 30% Teste) em ordem cronológica estrita.
7. Gerar todas as evidências e exemplos auditáveis para o diário da Tarefa 2.

> **Regras de Ouro:**
> - O Período A **nunca é rotulado** e **nunca entra no treino**.
> - RTT ausente é tratado como ausência (`NaN`), **nunca como zero**.
> - Nenhum limiar global fixo (como "RTT > 100 ms") ou identificador de rota (país, IP, probe_id) define falha nem pode entrar na árvore.


In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    caminho_colab = '/content/drive/MyDrive/project-redes'
    if os.path.exists(caminho_colab):
        os.chdir(caminho_colab)
    print("Ambiente detectado: Google Colab")
except ModuleNotFoundError:
    print("Ambiente detectado: Local (VS Code - Windows / Linux / macOS)")

    # Resolve para C:\Users\nome no Windows ou /home/nome no Linux
    user_home = os.path.expanduser("~")

    # Lista de locais comuns em qualquer Sistema Operacional
    caminhos_possiveis = [
        # Diretamente na pasta home (~/project-redes) - Muito comum no Linux
        os.path.join(user_home, "project-redes"),
        # Documentos (Windows com e sem OneDrive / Linux)
        os.path.join(user_home, "OneDrive", "Documentos", "project-redes"),
        os.path.join(user_home, "OneDrive", "Documents", "project-redes"),
        os.path.join(user_home, "Documentos", "project-redes"),
        os.path.join(user_home, "Documents", "project-redes"),
        # Outras pastas habituais de desenvolvimento
        os.path.join(user_home, "Downloads", "project-redes"),
        os.path.join(user_home, "Desktop", "project-redes"),
        os.path.join(user_home, "Área de Trabalho", "project-redes"),
        os.path.join(user_home, "projetos", "project-redes"),
        os.path.join(user_home, "projects", "project-redes"),
    ]

    caminho_encontrado = None
    for caminho in caminhos_possiveis:
        if os.path.exists(caminho):
            caminho_encontrado = caminho
            break

    if caminho_encontrado:
        os.chdir(caminho_encontrado)
        print(f"Diretório ajustado com sucesso para: {caminho_encontrado}")
    else:
        print("AVISO: A pasta 'project-redes' não foi encontrada nos caminhos padrão!")

print("Diretório de trabalho atual:", os.getcwd())

Ambiente detectado: Local (VS Code - Windows / Linux / macOS)
Diretório ajustado com sucesso para: C:\Users\vinic\OneDrive\Documentos\project-redes
Diretório de trabalho atual: C:\Users\vinic\OneDrive\Documentos\project-redes


In [ ]:
# ==============================================================================
# 1. IMPORTAÇÕES E CONFIGURAÇÕES GERAIS
# ==============================================================================
import os
import glob
import re
import json
import math
from datetime import datetime, timezone
import numpy as np
import pandas as pd

# Garantir compatibilidade com display em qualquer ambiente
try:
    from IPython.display import display
except Exception:
    display = print

# Fixar semente para reprodutibilidade
np.random.seed(42)

# Configurações de caminhos e parâmetros do projeto
PASTA_MEDICOES = os.path.join("metadados", "medicoes")          # Caminho relativo para a pasta com as medições
DIAS_PERIODO_A = 7                   # Duração do bloco de baseline em dias
SEGUNDOS_PERIODO_A = DIAS_PERIODO_A * 24 * 3600
PISO_MINIMO_AMOSTRAS_A = 1500        # Piso regulamentar de RTTs válidos no Período A

ARQUIVO_BASELINE = "baseline_por_fluxo.csv"
ARQUIVO_ROTULADO = "dataset_rotulado_periodo_b.csv"

print("Ambiente configurado com sucesso!")
print(f"Diretório de medições configurado: '{PASTA_MEDICOES}'")
print(f"Período A: {DIAS_PERIODO_A} dias ({SEGUNDOS_PERIODO_A:,} segundos) | Piso mínimo: {PISO_MINIMO_AMOSTRAS_A} amostras")


Ambiente configurado com sucesso!
Diretório de medições configurado: 'metadados\medicoes'
Período A: 7 dias (604,800 segundos) | Piso mínimo: 1500 amostras


## 1. Inspeção e Consolidação dos Dados Brutos

Carregamos todos os arquivos CSV presentes nas subpastas de `medicoes/`.
Para cada medição registrada:
- Criamos a chave canônica estável do fluxo: `fluxo_id = f"{prb_id}|{dst_addr}"`.
- RTT ausente ou timeout é mantido como `NaN`, preservando a perda real sem falsear estabilidade com zero.
- Calculamos o `jitter_ms` como o desvio-padrão amostral dos RTTs individuais da rajada quando há **2 ou mais respostas válidas**. Se houver apenas 1 resposta ou nenhuma, o jitter fica vazio (`NaN`).
- Calculamos a `perda_pct = (sent - rcvd) / sent * 100`.
- Sinalizamos `timeout_atual = 1` se não houve resposta (`rcvd == 0`) ou perda total.


In [ ]:
# ==============================================================================
# 2. CARGA DOS DADOS BRUTOS E ENGENHARIA DE CAMPOS INICIAIS
# ==============================================================================
pattern_rtt = re.compile(r"'rtt':\s*([0-9.]+)")

def extrair_jitter(result_str):
    """
    Calcula o desvio-padrão amostral dos RTTs da rajada apenas quando há >= 2 respostas.
    Caso contrário, retorna NaN (nunca 0 ms para não simular estabilidade inexistente).
    """
    if not isinstance(result_str, str):
        return np.nan
    vals = [float(x) for x in pattern_rtt.findall(result_str)]
    if len(vals) >= 2:
        mean_val = sum(vals) / len(vals)
        variance = sum((x - mean_val) ** 2 for x in vals) / (len(vals) - 1)
        return math.sqrt(variance)
    return np.nan

csv_files = sorted(glob.glob(os.path.join(PASTA_MEDICOES, "*", "*.csv")))
print(f"Encontrados {len(csv_files)} arquivos CSV em '{PASTA_MEDICOES}':")
for f in csv_files:
    print(" -", f)

dfs = []
for f in csv_files:
    sub_df = pd.read_csv(f)
    dfs.append(sub_df)

df_raw = pd.concat(dfs, ignore_index=True)
print(f"\nTotal de registros brutos carregados: {len(df_raw):,}")

# Identificador estável do fluxo
df_raw["prb_id"] = df_raw["prb_id"].astype(str)
df_raw["dst_addr"] = df_raw["dst_addr"].astype(str)
df_raw["fluxo_id"] = df_raw["prb_id"] + "|" + df_raw["dst_addr"]

# RTT médio (ms): se ausente, nulo ou <= 0, vira NaN
df_raw["rtt_ms"] = pd.to_numeric(df_raw["avg"], errors="coerce")
df_raw.loc[df_raw["rtt_ms"] <= 0, "rtt_ms"] = np.nan

# Pacotes enviados e recebidos
df_raw["sent"] = pd.to_numeric(df_raw["sent"], errors="coerce").fillna(3)
df_raw["rcvd"] = pd.to_numeric(df_raw["rcvd"], errors="coerce").fillna(0)

# Perda percentual da rajada
df_raw["perda_pct"] = np.where(
    df_raw["sent"] > 0,
    ((df_raw["sent"] - df_raw["rcvd"]) / df_raw["sent"]) * 100.0,
    100.0
)
df_raw["perda_pct"] = df_raw["perda_pct"].clip(lower=0.0, upper=100.0)

# Timeout atual da medição (1 se sem resposta ou perda=100%, senão 0)
df_raw["timeout_atual"] = np.where(
    (df_raw["rcvd"] == 0) | (df_raw["rtt_ms"].isna()) | (df_raw["perda_pct"] >= 100.0),
    1,
    0
)

# Jitter da rajada (ms)
print("Extraindo jitter individual de cada rajada...")
df_raw["jitter_ms"] = df_raw["result"].apply(extrair_jitter)

# Timestamp e ordenação temporal rigorosa
df_raw["timestamp"] = pd.to_numeric(df_raw["timestamp"], errors="coerce").astype(int)
df_raw = df_raw.sort_values(by=["fluxo_id", "timestamp"]).reset_index(drop=True)

# Auditoria inicial
n_fluxos = df_raw["fluxo_id"].nunique()
n_timeouts = df_raw["timeout_atual"].sum()
dt_min = datetime.fromtimestamp(df_raw["timestamp"].min(), timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
dt_max = datetime.fromtimestamp(df_raw["timestamp"].max(), timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

print("\n--- RELATÓRIO DE QUALIDADE DOS DADOS BRUTOS ---")
print(f"Total de medições: {len(df_raw):,}")
print(f"Total de fluxos únicos: {n_fluxos}")
print(f"Janela temporal: {dt_min} até {dt_max}")
print(f"Timeouts detectados: {n_timeouts:,} ({n_timeouts/len(df_raw)*100:.2f}%)")
print(f"Medições com RTT válido: {df_raw['rtt_ms'].notna().sum():,}")
print(f"Medições com Jitter calculado: {df_raw['jitter_ms'].notna().sum():,}")


Encontrados 7 arquivos CSV em 'metadados\medicoes':
 - metadados\medicoes\109063401\ripe_atlas_m109063401_20261015T012758Z.csv
 - metadados\medicoes\109069241\ripe_atlas_m109069241_20261015T012859Z.csv
 - metadados\medicoes\199622940\ripe_atlas_m199622940_20261015T013437Z.csv
 - metadados\medicoes\28095697\ripe_atlas_m28095697_20261015T012134Z.csv
 - metadados\medicoes\28097617\ripe_atlas_m28097617_20261015T012248Z.csv
 - metadados\medicoes\33008006\ripe_atlas_m33008006_20261015T014150Z.csv
 - metadados\medicoes\33008382\ripe_atlas_m33008382_20261015T014252Z.csv

Total de registros brutos carregados: 189,192
Extraindo jitter individual de cada rajada...

--- RELATÓRIO DE QUALIDADE DOS DADOS BRUTOS ---
Total de medições: 189,192
Total de fluxos únicos: 35
Janela temporal: 2026-09-29 19:23 UTC até 2026-10-15 01:39 UTC
Timeouts detectados: 471 (0.25%)
Medições com RTT válido: 188,721
Medições com Jitter calculado: 188,403


## 2. Separação Estrita dos Períodos A e B

Conforme a especificação do projeto:
- **Período A (Baseline):** 7 dias exatos a partir do primeiro timestamp do fluxo.
- **Período B (Rotulagem):** a partir do corte exato de 7 dias em diante.
- Não há compartilhamento de amostras nem de timestamps entre os dois blocos para o mesmo fluxo.


In [ ]:
# ==============================================================================
# 3. DIVISÃO CONTÍGUA DOS PERÍODOS A E B (SEM SOBREPOSIÇÃO)
# ==============================================================================
# Calculamos o corte exato para cada fluxo (início + 7 dias)
corte_por_fluxo = df_raw.groupby("fluxo_id")["timestamp"].min() + SEGUNDOS_PERIODO_A
df_raw["timestamp_corte"] = df_raw["fluxo_id"].map(corte_por_fluxo)

df_periodo_a = df_raw[df_raw["timestamp"] < df_raw["timestamp_corte"]].copy()
df_periodo_b = df_raw[df_raw["timestamp"] >= df_raw["timestamp_corte"]].copy()

# Validação formal de não-sobreposição
max_a = df_periodo_a.groupby("fluxo_id")["timestamp"].max()
min_b = df_periodo_b.groupby("fluxo_id")["timestamp"].min()
sobreposicao = (max_a >= min_b).any()

# Salvar os blocos brutos separados em CSV para persistência e auditoria
df_periodo_a.to_csv("periodo_a_bruto.csv", index=False)
df_periodo_b.to_csv("periodo_b_bruto.csv", index=False)

print("--- AUDITORIA DA SEPARAÇÃO TEMPORAL ---")
print(f"Período A (Baseline congelado): {len(df_periodo_a):,} registros (salvo em 'periodo_a_bruto.csv')")
print(f"Período B (Rotulagem):          {len(df_periodo_b):,} registros (salvo em 'periodo_b_bruto.csv')")
print(f"Existe sobreposição temporal no mesmo fluxo? {'SIM [ERRO]' if sobreposicao else 'NÃO [CORRETO ✅]'}")


--- AUDITORIA DA SEPARAÇÃO TEMPORAL ---
Período A (Baseline congelado): 87,243 registros (salvo em 'periodo_a_bruto.csv')
Período B (Rotulagem):          101,949 registros (salvo em 'periodo_b_bruto.csv')
Existe sobreposição temporal no mesmo fluxo? NÃO [CORRETO ✅]


## 3. Construção da Ficha de Baseline (`baseline_por_fluxo.csv`)

A ficha é calculada **exclusivamente com o Período A** e congelada:
1. `mediana`: mediana dos RTTs válidos (ms) — timeouts não entram na mediana.
2. `MAD`: mediana(|RTT - mediana|).
3. `IQR`: $Q_{75} - Q_{25}$ (utilizado caso MAD = 0 e RTT $
eq$ mediana).
4. `jitter_tipico`: mediana do jitter nas medições em que o jitter existe (ms).
5. `perda_tipica`: mediana de `perda_pct`, inclusive timeout (%).
6. `prop_resposta`: medições com RTT válido / total de medições no Período A.
7. **Piso de Amostras:** fluxos com menos de 1.500 RTTs válidos no Período A recebem `baseline_insuficiente` e são excluídos da rotulagem.


In [ ]:
# ==============================================================================
# 4. GERAÇÃO DA FICHA DE BASELINE POR FLUXO (SOMENTE PERÍODO A)
# ==============================================================================
fichas = []
fluxos_excluidos = []

for fluxo_id, df_fluxo_a in df_periodo_a.groupby("fluxo_id"):
    total_obs = len(df_fluxo_a)
    rtts_validos = df_fluxo_a["rtt_ms"].dropna()
    jitters_validos = df_fluxo_a["jitter_ms"].dropna()

    n_validos = len(rtts_validos)

    if n_validos < PISO_MINIMO_AMOSTRAS_A:
        fluxos_excluidos.append({
            "fluxo_id": fluxo_id,
            "amostras_validas_A": n_validos,
            "motivo": f"baseline_insuficiente (< {PISO_MINIMO_AMOSTRAS_A})"
        })
        continue

    mediana_rtt = float(rtts_validos.median())
    mad_rtt = float((rtts_validos - mediana_rtt).abs().median())

    q25 = float(rtts_validos.quantile(0.25))
    q75 = float(rtts_validos.quantile(0.75))
    iqr_rtt = q75 - q25

    jitter_tipico = float(jitters_validos.median()) if len(jitters_validos) > 0 else 0.0
    perda_tipica = float(df_fluxo_a["perda_pct"].median())
    prop_resposta = float(n_validos / total_obs)

    fichas.append({
        "fluxo_id": fluxo_id,
        "probe_id": df_fluxo_a["prb_id"].iloc[0],
        "dst_addr": df_fluxo_a["dst_addr"].iloc[0],
        "total_obs_A": total_obs,
        "validos_A": n_validos,
        "mediana": round(mediana_rtt, 3),
        "MAD": round(mad_rtt, 4),
        "IQR": round(iqr_rtt, 4),
        "jitter_tipico": round(jitter_tipico, 4),
        "perda_tipica": round(perda_tipica, 2),
        "prop_resposta": round(prop_resposta, 4)
    })

df_baseline = pd.DataFrame(fichas)
df_baseline.to_csv(ARQUIVO_BASELINE, index=False)
print(f"Ficha de baseline gerada e congelada com sucesso em '{ARQUIVO_BASELINE}'!")
print(f"Total de fluxos com baseline aprovado: {len(df_baseline)}")
print(f"Total de fluxos excluídos por piso (<1500): {len(fluxos_excluidos)}")

# Exibição de amostra auditável (fluxo curto vs fluxo longo)
df_baseline_sorted = df_baseline.sort_values(by="mediana")
ex_curto = df_baseline_sorted.iloc[0]
ex_longo = df_baseline_sorted.iloc[-1]

print("\n--- EXEMPLOS AUDITÁVEIS DE BASELINE (INDEPENDÊNCIA DA ROTA) ---")
print(f"Fluxo Curto:  [{ex_curto['fluxo_id']}] -> Mediana = {ex_curto['mediana']} ms | MAD = {ex_curto['MAD']} ms | Amostras A = {ex_curto['validos_A']}")
print(f"Fluxo Longo:  [{ex_longo['fluxo_id']}] -> Mediana = {ex_longo['mediana']} ms | MAD = {ex_longo['MAD']} ms | Amostras A = {ex_longo['validos_A']}")
print("> Note que ambas as medianas representam o estado 'NORMAL' de seus respectivos caminhos!")

display(df_baseline.head(10))


Ficha de baseline gerada e congelada com sucesso em 'baseline_por_fluxo.csv'!
Total de fluxos com baseline aprovado: 35
Total de fluxos excluídos por piso (<1500): 0

--- EXEMPLOS AUDITÁVEIS DE BASELINE (INDEPENDÊNCIA DA ROTA) ---
Fluxo Curto:  [1013398|217.25.179.62] -> Mediana = 1.692 ms | MAD = 0.0782 ms | Amostras A = 2519
Fluxo Longo:  [6756|103.96.67.60] -> Mediana = 413.153 ms | MAD = 72.8095 ms | Amostras A = 2279
> Note que ambas as medianas representam o estado 'NORMAL' de seus respectivos caminhos!


,fluxo_id,probe_id,dst_addr,total_obs_A,validos_A,mediana,MAD,IQR,jitter_tipico,perda_tipica,prop_resposta
0,1001323|103.96.67.60,1001323,103.96.67.60,1788,1787,22.250,0.2302,0.4612,0.4414,0.0,0.9994
1,1004498|103.96.67.60,1004498,103.96.67.60,2466,2465,17.432,0.2047,0.4077,0.3870,0.0,0.9996
2,1013398|217.25.179.62,1013398,217.25.179.62,2519,2519,1.692,0.0782,0.1604,0.1131,0.0,1.0000
3,1013813|150.164.1.222,1013813,150.164.1.222,2517,2513,4.634,0.0684,0.1453,0.0881,0.0,0.9984
4,1016453|217.25.179.62,1016453,217.25.179.62,2519,2519,9.163,0.2382,0.4820,0.4883,0.0,1.0000
5,10836|150.164.1.222,10836,150.164.1.222,2521,2521,205.655,0.5406,1.6420,0.0959,0.0,1.0000
6,15391|103.96.67.60,15391,103.96.67.60,2521,2521,178.028,4.6530,9.2302,0.4622,0.0,1.0000
7,18755|217.25.179.62,18755,217.25.179.62,2520,2520,7.406,0.0606,0.1269,0.0721,0.0,1.0000
8,20423|150.164.1.222,20423,150.164.1.222,2521,2516,134.944,1.4859,3.4012,0.1579,0.0,0.9980
9,2629|217.25.179.62,2629,217.25.179.62,2518,2517,20.035,0.3118,0.6664,0.4032,0.0,0.9996


## 4. Cálculo das Métricas Relativas no Período B

Utilizando exclusivamente os parâmetros congelados na ficha de baseline de cada fluxo:
- `z_robusto` $= \frac{\text{RTT atual} - \text{mediana}}{1{,}4826 \times \text{MAD}}$
  - Se RTT ausente: não calcular (`NaN`). `timeout_atual = 1`.
  - Se MAD $= 0$ e RTT atual $=$ mediana: `z_robusto = 0.0`.
  - Se MAD $= 0$ e RTT atual $
eq$ mediana: denominador $= \max(\text{IQR} / 1{,}349; 1{,}0\text{ ms})$.
- `aumento_pct` $= \frac{\text{RTT atual} - \text{mediana}}{\text{mediana}} \times 100$
- `jitter_relativo` $= \frac{\text{jitter atual}}{\text{jitter\_tipico}}$
  - Se jitter atual vazio: não dispara.
  - Se `jitter_tipico == 0` e jitter atual $= 0$: $1{,}0$.
  - Se `jitter_tipico == 0` e jitter atual $> 0$: tratar como $\ge 3{,}0$.
- **Janelas deslizantes das últimas 5 medições (incluindo a atual, por fluxo):**
  - `n5_timeout`: contagem de timeouts nas últimas 5 medições.
  - `n5_aumento80`: quantas das últimas 5 têm `aumento_pct > 80%`.
  - `n5_risco`: quantas das últimas 5 cumprem a condição individual de risco.


In [ ]:
# ==============================================================================
# 5. CÁLCULO DAS MÉTRICAS RELATIVAS E JANELAS DESLIZANTES EM B
# ==============================================================================
# Filtrar apenas os fluxos que possuem ficha de baseline válida
df_b_validos = df_periodo_b[df_periodo_b["fluxo_id"].isin(df_baseline["fluxo_id"])].copy()

# Fazer merge com a ficha de baseline congelada
df_b = df_b_validos.merge(
    df_baseline[["fluxo_id", "mediana", "MAD", "IQR", "jitter_tipico", "perda_tipica"]],
    on="fluxo_id",
    how="left"
)

# 1. z_robusto
def calcular_z_robusto(row):
    rtt = row["rtt_ms"]
    med = row["mediana"]
    mad = row["MAD"]
    iqr = row["IQR"]

    if pd.isna(rtt):
        return np.nan
    if mad == 0:
        if rtt == med:
            return 0.0
        else:
            denominador = max(iqr / 1.349, 1.0)
            return (rtt - med) / denominador
    return (rtt - med) / (1.4826 * mad)

df_b["z_robusto"] = df_b.apply(calcular_z_robusto, axis=1)

# 2. aumento_pct
df_b["aumento_pct"] = np.where(
    df_b["rtt_ms"].notna(),
    ((df_b["rtt_ms"] - df_b["mediana"]) / df_b["mediana"]) * 100.0,
    np.nan
)

# 3. jitter_relativo
def calcular_jitter_relativo(row):
    j_atual = row["jitter_ms"]
    j_tipico = row["jitter_tipico"]

    if pd.isna(j_atual):
        return np.nan
    if j_tipico == 0:
        return 1.0 if j_atual == 0 else 3.0
    return j_atual / j_tipico

df_b["jitter_relativo"] = df_b.apply(calcular_jitter_relativo, axis=1)

# Condição individual de risco (nesta medição)
# Pelo menos um: (2 <= z_robusto < 3.5) OU (30 <= aumento_pct <= 80) OU (jitter_relativo >= 3)
cond_z = (df_b["z_robusto"] >= 2.0) & (df_b["z_robusto"] < 3.5)
cond_aumento = (df_b["aumento_pct"] >= 30.0) & (df_b["aumento_pct"] <= 80.0)
cond_jitter = (df_b["jitter_relativo"] >= 3.0)

df_b["cond_risco_ind"] = (cond_z | cond_aumento | cond_jitter).astype(int)
df_b["cond_aumento80"] = (df_b["aumento_pct"] > 80.0).astype(int)

# 4. Janelas móveis de 5 medições por fluxo em ordem temporal
df_b = df_b.sort_values(by=["fluxo_id", "timestamp"]).reset_index(drop=True)

df_b["n5_timeout"] = df_b.groupby("fluxo_id")["timeout_atual"].rolling(5, min_periods=1).sum().reset_index(drop=True).astype(int)
df_b["n5_aumento80"] = df_b.groupby("fluxo_id")["cond_aumento80"].rolling(5, min_periods=1).sum().reset_index(drop=True).astype(int)
df_b["n5_risco"] = df_b.groupby("fluxo_id")["cond_risco_ind"].rolling(5, min_periods=1).sum().reset_index(drop=True).astype(int)

print("Métricas relativas e janelas de persistência calculadas com sucesso no Período B!")
display(df_b[["fluxo_id", "timestamp", "rtt_ms", "mediana", "z_robusto", "aumento_pct", "jitter_relativo", "n5_timeout", "n5_risco"]].head(5))


Métricas relativas e janelas de persistência calculadas com sucesso no Período B!


,fluxo_id,timestamp,rtt_ms,mediana,z_robusto,aumento_pct,jitter_relativo,n5_timeout,n5_risco
0,1001323|103.96.67.60,1791677692,22.437884,22.25,0.550504,0.844422,0.308909,0,0
1,1001323|103.96.67.60,1791677927,22.080410,22.25,-0.496902,-0.762202,1.200390,0,0
2,1001323|103.96.67.60,1791678162,21.620927,22.25,-1.843197,-2.827294,1.749022,0,0
3,1001323|103.96.67.60,1791678413,22.336971,22.25,0.254828,0.390882,0.953374,0,0
4,1001323|103.96.67.60,1791678651,22.701010,22.25,1.321469,2.027011,0.530969,0,0


## 5. Tabela de Rotulagem — Parar na Primeira Linha Verdadeira

Para cada linha do Período B, aplicamos a hierarquia oficial da rubrica:

| Ordem | Classe | Métrica | Limiar exato |
| :---: | :---: | :--- | :--- |
| **1** | **FALHA** | `perda_pct` | $\ge 10\%$ nesta medição (em 3 pacotes, 1 perda já é 33,3%) |
| **2** | **FALHA** | `n5_timeout` | $\ge 3$ timeouts nas últimas 5 medições |
| **3** | **FALHA** | `z_robusto` | $\ge 3{,}5$ nesta medição, com RTT presente |
| **4** | **FALHA** | `n5_aumento80` | $\ge 2$ medições com aumento $> 80\%$ nas últimas 5 |
| **5** | **RISCO** | `cond_risco_ind` e `n5_risco` | Condição de risco nesta medição E persistência `n5_risco` $\ge 2$ |
| **6** | **OK** | Nenhuma das anteriores | Picos transitórios isolados sem persistência também são OK |


In [ ]:
# ==============================================================================
# 6. ROTULAGEM AUTOMÁTICA DAS CLASSES EM B
# ==============================================================================
def rotular_medicao(row):
    """
    Define a classe e o motivo do rótulo para cada medição no Período B
    com base nas métricas relativas e janelas deslizantes de 5 amostras.
    """
    # 1. Condição de Falha / Queda Severa
    if row["n5_timeout"] >= 3 or row["perda_pct"] >= 10.0:
        motivo = "n5_timeout >= 3" if row["n5_timeout"] >= 3 else "perda_pct >= 10"
        return "FALHA", motivo

    # 2. Condição de Degradacao / Risco
    if row["z_robusto"] >= 3.5:
        return "RISCO", "z_robusto >= 3.5"
    if row["n5_risco"] >= 2:
        return "RISCO", "risco_persistente (n5_risco >= 2)"
    if row["n5_aumento80"] >= 2:
        return "RISCO", "n5_aumento80 >= 2"

    # 3. Estado Normal
    return "OK", "normal / pico_isolado"

# Aplicação da regra de rotulagem no DataFrame df_b
rotulos = df_b.apply(rotular_medicao, axis=1)
df_b["classe"] = [r[0] for r in rotulos]
df_b["motivo_rotulo"] = [r[1] for r in rotulos]

print("--- DISTRIBUIÇÃO DAS CLASSES ROTULADAS EM B ---")
print(df_b["classe"].value_counts())
print("\n--- DETALHAMENTO DOS MOTIVOS ---")
print(df_b["motivo_rotulo"].value_counts())

--- DISTRIBUIÇÃO DAS CLASSES ROTULADAS EM B ---
classe
OK       80090
RISCO    21373
FALHA      486
Name: count, dtype: int64

--- DETALHAMENTO DOS MOTIVOS ---
motivo_rotulo
normal / pico_isolado                80090
z_robusto >= 3.5                     13188
risco_persistente (n5_risco >= 2)     8170
perda_pct >= 10                        399
n5_timeout >= 3                         87
n5_aumento80 >= 2                       15
Name: count, dtype: int64


## 6. Exemplos Auditáveis para o Diário de Bordo

O contrato da Tarefa 2 exige três exemplos reais demonstrando:
1. **OK em caminho longo:** RTT absoluto elevado (ex: > 150 ms), mas `z_robusto` baixo ($pprox 0$), provando que rota longa **não é falha**.
2. **RISCO:** medição com degradação moderada persistente (`n5_risco >= 2`).
3. **FALHA:** medição que disparou perda $\ge 10\%$, timeout persistente ou $z \ge 3{,}5$.


In [ ]:
# ==============================================================================
# 7. DIVISÃO TEMPORAL DO DATASET EM B (70% Treino / 15% Validação / 15% Teste)
# ==============================================================================

# 1. Ordenação temporal obrigatória por timestamp para evitar contaminação
df_b = df_b.sort_values(by="timestamp").reset_index(drop=True)

# 2. Cálculo dos pontos de corte exatos
n_total = len(df_b)
idx_treino = int(n_total * 0.70)
idx_val = int(n_total * 0.85)

# 3. Atribuição direta e explícita da coluna no DataFrame principal
df_b["split_temporal"] = "teste"
df_b.loc[:idx_treino - 1, "split_temporal"] = "treino"
df_b.loc[idx_treino:idx_val - 1, "split_temporal"] = "validacao"

print("--- VERIFICAÇÃO DA SEÇÃO 7 ---")
print("Coluna 'split_temporal' criada e atribuída com sucesso ao df_b!")
print(df_b["split_temporal"].value_counts())

--- VERIFICAÇÃO DA SEÇÃO 7 ---
Coluna 'split_temporal' criada e atribuída com sucesso ao df_b!
split_temporal
treino       71364
teste        15293
validacao    15292
Name: count, dtype: int64


## 7. Recorte Temporal para a Futura Árvore

Conforme o contrato da Tarefa 2:
- O recorte é feito **estritamente dentro do Período B**, por fluxo, em ordem temporal.
- Proporção: **50% Treino** (bloco mais antigo) | **20% Validação** (bloco intermediário) | **30% Teste** (bloco mais recente).
- Nenhum registro do Período A entra no treino.


In [ ]:
# ==============================================================================
# 8. EXPORTAÇÃO DO DATASET FINAL ROTULADO E DIVIDIDO
# ==============================================================================
# Salva o df_b mantendo todas as colunas geradas ('classe', 'motivo_rotulo', 'split_temporal')
nome_arquivo_final = "dataset_rotulado_periodo_b.csv"
df_b.to_csv(nome_arquivo_final, index=False)

print("\n--- VERIFICAÇÃO DA SEÇÃO 8 ---")
print(f"Ficheiro '{nome_arquivo_final}' guardado com sucesso!")
print(f"Total de registos salvos: {len(df_b):,}")
print(f"Total de colunas salvas: {len(df_b.columns)}")


--- VERIFICAÇÃO DA SEÇÃO 8 ---
Ficheiro 'dataset_rotulado_periodo_b.csv' guardado com sucesso!
Total de registos salvos: 101,949
Total de colunas salvas: 48


## 8. Dicionário v0.2 e Colunas Proibidas na Árvore

### Colunas Permitidas para as Features da Árvore (Métricas Relativas):
- `z_robusto` (normalização pelo próprio baseline)
- `aumento_pct` (variação percentual de latência)
- `jitter_relativo` (desvio em relação à estabilidade típica)
- `perda_pct` (perda na rajada atual)
- `timeout_atual` (0 ou 1)
- `n5_timeout` (persistência de timeouts recentes)
- `n5_aumento80` (persistência de latência crítica)
- `n5_risco` (persistência de degradação moderada)

### 🚫 Colunas Proibidas na Entrada da Árvore de Decisão:
1. `dst_addr`, `prb_id`, `fluxo_id`, `rota_id` (impedem vazamento de identidade da rota).
2. `rtt_ms`, `min`, `max`, `avg` (o RTT absoluto impediria a generalização entre caminhos curtos e longos).
3. `timestamp`, `stored_timestamp` (impedem overfitting temporal).
4. `pais`, `continente` (não viram rótulo nem feature).


In [ ]:
# ==============================================================================
# 9. RESUMO CONSOLIDADO PARA COPIAR NO DIÁRIO DA TAREFA 2
# ==============================================================================
from datetime import datetime, timezone

# Formatação das datas do Período B
t_min_b = datetime.fromtimestamp(df_b["timestamp"].min(), timezone.utc).strftime("%d/%m/%Y %H:%M")
t_max_b = datetime.fromtimestamp(df_b["timestamp"].max(), timezone.utc).strftime("%d/%m/%Y %H:%M")

# Totais das contagens reais da divisão temporal
total_b = len(df_b)
c_treino = (df_b["split_temporal"] == "treino").sum()
c_val = (df_b["split_temporal"] == "validacao").sum()
c_teste = (df_b["split_temporal"] == "teste").sum()

# Percentuais reais calculados dinamicamente
pct_treino_real = (c_treino / total_b) * 100
pct_val_real = (c_val / total_b) * 100
pct_teste_real = (c_teste / total_b) * 100

# Percentuais de preferência projetados
pref_treino = 70.0
pref_val = 15.0
pref_teste = 15.0

# Contagem e percentual das classes de qualidade
n_ok = (df_b["classe"] == "OK").sum()
pct_ok = (n_ok / total_b) * 100

n_risco = (df_b["classe"] == "RISCO").sum()
pct_risco = (n_risco / total_b) * 100

n_falha = (df_b["classe"] == "FALHA").sum()
pct_falha = (n_falha / total_b) * 100

# Mapeamento seguro das variáveis dos blocos anteriores
len_raw = len(df_raw) if 'df_raw' in locals() else total_b
len_a = len(df_periodo_a) if 'df_periodo_a' in locals() else 0
len_base = len(df_baseline) if 'df_baseline' in locals() else 35
len_excl = len(fluxos_excluidos) if 'fluxos_excluidos' in locals() else 0

print(f"""
==============================================================================
RESUMO PARA O DIÁRIO DA TAREFA 2:
==============================================================================
- Total de registros brutos: {len_raw:,}
- Registros no Período A (Baseline congelado): {len_a:,}
- Registros no Período B (Rotulagem): {total_b:,}
- Fluxos com ficha de baseline aprovada: {len_base} (piso de 1.500 validado em todos)
- Fluxos excluídos: {len_excl}

- Distribuição de Classes no Período B:
  - OK:    {n_ok:,} ({pct_ok:.2f}%)
  - RISCO: {n_risco:,} ({pct_risco:.2f}%)
  - FALHA: {n_falha:,} ({pct_falha:.2f}%)

- Recorte Temporal em B ({t_min_b} até {t_max_b}):
  - Treino:
      Preferência: {pref_treino:.0f}%
      Real:        {pct_treino_real:.2f}% ({c_treino:,} amostras)
  - Validação:
      Preferência: {pref_val:.0f}%
      Real:        {pct_val_real:.2f}% ({c_val:,} amostras)
  - Teste:
      Preferência: {pref_teste:.0f}%
      Real:        {pct_teste_real:.2f}% ({c_teste:,} amostras)
==============================================================================
""")


RESUMO PARA O DIÁRIO DA TAREFA 2:
- Total de registros brutos: 189,192
- Registros no Período A (Baseline congelado): 87,243
- Registros no Período B (Rotulagem): 101,949
- Fluxos com ficha de baseline aprovada: 35 (piso de 1.500 validado em todos)
- Fluxos excluídos: 0

- Distribuição de Classes no Período B:
  - OK:    80,090 (78.56%)
  - RISCO: 21,373 (20.96%)
  - FALHA: 486 (0.48%)

- Recorte Temporal em B (06/10/2026 19:23 até 15/10/2026 01:39):
  - Treino:
      Preferência: 70%
      Real:        70.00% (71,364 amostras)
  - Validação:
      Preferência: 15%
      Real:        15.00% (15,292 amostras)
  - Teste:
      Preferência: 15%
      Real:        15.00% (15,293 amostras)

